In [6]:
from pathlib import Path
import json
import platform
import subprocess
import sys

PROJECT = Path("/home/jovyan/project/georgia-bid-readiness")

if not PROJECT.is_dir():
    raise FileNotFoundError(f"Project folder not found: {PROJECT}")

folders = [
    "configs",
    "data/raw",
    "data/processed",
    "data/labels",
    "notebooks",
    "src/ingestion",
    "src/extraction",
    "src/transformation",
    "src/evaluation",
    "warehouse/ddl",
    "warehouse/models",
    "warehouse/tests",
    "app",
    "experiments/runs",
    "experiments/receipts",
    "reports",
    "logs",
]

for folder in folders:
    path = PROJECT / folder
    path.mkdir(parents=True, exist_ok=True)

    # Git does not track empty directories.
    (path / ".gitkeep").touch(exist_ok=True)

def create_if_missing(relative_path, content):
    path = PROJECT / relative_path
    if path.exists():
        print("Kept existing:", relative_path)
    else:
        path.write_text(content, encoding="utf-8")
        print("Created:", relative_path)

config = {
    "project_name": "Georgia Bid Readiness",
    "config_version": 1,
    "paths": {
        "raw": "data/raw",
        "processed": "data/processed",
        "labels": "data/labels",
        "logs": "logs",
        "runs": "experiments/runs",
        "receipts": "experiments/receipts",
    },
    "database": {
        "engine": "postgresql",
        "connection_env_var": "GBR_DATABASE_URL",
        "hosting_status": "not_configured",
    },
}

create_if_missing(
    "configs/project.json",
    json.dumps(config, indent=2) + "\n",
)

create_if_missing(
    ".gitignore",
    """# Secrets
.env
.env.*
!.env.example
.streamlit/secrets.toml

# Python and notebook temporary files
__pycache__/
*.py[cod]
.ipynb_checkpoints/
.venv/
venv/

# Data and run outputs are backed up separately
data/raw/**
!data/raw/.gitkeep
data/processed/**
!data/processed/.gitkeep
data/labels/**
!data/labels/.gitkeep
experiments/runs/**
!experiments/runs/.gitkeep
experiments/receipts/**
!experiments/receipts/.gitkeep
logs/**
!logs/.gitkeep

# Large model files and archives
*.safetensors
*.pt
*.pth
*.zip
.cache/
checkpoints/
""",
)

create_if_missing(
    ".env.example",
    "# Set this when PostgreSQL hosting is configured.\n"
    "GBR_DATABASE_URL=\n",
)

create_if_missing(
    "reports/experiment_register.csv",
    "run_id,experiment,date_utc,model,dataset_version,"
    "prompt_version,status,result_file,receipt_file,notes\n",
)

# Capture the CPU environment without installing or upgrading packages.
snapshot = {
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "python_executable": sys.executable,
}

create_if_missing(
    "configs/cpu_environment_initial.json",
    json.dumps(snapshot, indent=2) + "\n",
)

freeze = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    capture_output=True,
    text=True,
    check=True,
)

create_if_missing(
    "configs/cpu_packages_initial.txt",
    freeze.stdout,
)

print("\nProject structure ready:", PROJECT)
print("README present:", (PROJECT / "README.md").is_file())

Kept existing: configs/project.json
Kept existing: .gitignore
Kept existing: .env.example
Kept existing: reports/experiment_register.csv
Kept existing: configs/cpu_environment_initial.json
Kept existing: configs/cpu_packages_initial.txt

Project structure ready: /home/jovyan/project/georgia-bid-readiness
README present: True


In [7]:
import importlib.metadata as metadata
import shutil

for package in [
    "pypdf",
    "pandas",
    "requests",
    "sqlalchemy",
    "psycopg",
    "dbt-core",
    "streamlit",
]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("\nCommand-line tools:")
for command in ["git", "tesseract", "ocrmypdf", "psql", "postgres"]:
    print(f"{command}: {shutil.which(command) or 'NOT FOUND'}")

pypdf: 6.21.0
pandas: 2.1.1
requests: 2.34.2
sqlalchemy: 2.0.22
psycopg: NOT INSTALLED
dbt-core: NOT INSTALLED
streamlit: NOT INSTALLED

Command-line tools:
git: /usr/bin/git
tesseract: NOT FOUND
ocrmypdf: NOT FOUND
psql: NOT FOUND
postgres: NOT FOUND


In [8]:
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED
import hashlib

BACKUPS = PROJECT.parent / "georgia-bid-readiness-backups"
BACKUPS.mkdir(exist_ok=True)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
archive_path = BACKUPS / f"georgia-bid-readiness_{stamp}.zip"

excluded_dirs = {
    ".git", ".venv", "venv", "__pycache__",
    ".ipynb_checkpoints", ".cache", "checkpoints",
}

def include_file(path):
    relative = path.relative_to(PROJECT)

    if path.is_symlink():
        return False
    if any(part in excluded_dirs for part in relative.parts):
        return False
    if path.name.startswith(".env") and path.name != ".env.example":
        return False
    if path.name == "secrets.toml":
        return False
    return path.suffix.lower() not in {
        ".pyc", ".safetensors", ".pt", ".pth", ".zip"
    }

def file_hash(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

manifest = {}

with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    for path in sorted(PROJECT.rglob("*")):
        if path.is_file() and include_file(path):
            name = path.relative_to(PROJECT.parent).as_posix()
            manifest[name] = file_hash(path)
            archive.write(path, arcname=name)

with ZipFile(archive_path) as archive:
    if archive.testzip() is not None:
        raise RuntimeError("Backup integrity check failed.")

    for name, expected in manifest.items():
        digest = hashlib.sha256()
        with archive.open(name) as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)
        if digest.hexdigest() != expected:
            raise RuntimeError(f"Backup mismatch: {name}")

print("Backup verified:", archive_path)
print("Files included:", len(manifest))
print("Archive size MB:", round(archive_path.stat().st_size / 1024**2, 2))

Backup verified: /home/jovyan/project/georgia-bid-readiness-backups/georgia-bid-readiness_20261016T221603870182Z.zip
Files included: 29
Archive size MB: 0.02


In [4]:
%pip install --user pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 242.1 kB/s eta 0:00:000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [5]:
import importlib.metadata as metadata
from pathlib import Path
from pypdf import PdfReader

PROJECT = Path("/home/jovyan/project/georgia-bid-readiness")

version = metadata.version("pypdf")
requirements = PROJECT / "requirements-cpu.txt"

# Preserve any existing requirements.
existing = (
    requirements.read_text(encoding="utf-8")
    if requirements.exists()
    else ""
)

if not any(
    line.strip().lower().startswith("pypdf")
    for line in existing.splitlines()
):
    with requirements.open("a", encoding="utf-8") as file:
        if existing and not existing.endswith("\n"):
            file.write("\n")
        file.write(f"pypdf=={version}\n")

print("PDF support ready:", version)
print("Requirements saved:", requirements)

PDF support ready: 6.21.0
Requirements saved: /home/jovyan/project/georgia-bid-readiness/requirements-cpu.txt
